# Population Analysis — Italian Real Estate Market

Build a reproducible **municipality-year population panel** from the ISTAT POSAS releases stored in the repository.

**Pipeline:** raw population-by-age files → inventory → validation → municipality totals → demographic indicators → geographic analysis → analytical hand-off.

The municipality files contain one row per age and an official `Età = 999` total row.


## 1. Setup and source inventory

Years and files are discovered automatically so the notebook remains reproducible when a new annual release is added.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
assert RAW_DIR.exists(), f'Population directory not found: {RAW_DIR}'

YEAR_DIRS = sorted(p for p in RAW_DIR.iterdir() if p.is_dir() and p.name.isdigit())
YEARS = [int(p.name) for p in YEAR_DIRS]
print(f'Population releases: {min(YEARS)}–{max(YEARS)}')
print(f'Years available: {YEARS}')

inventory = []
for folder in YEAR_DIRS:
    files = sorted(folder.glob('*.csv'))
    inventory.append({
        'year': int(folder.name),
        'csv_files': len(files),
        'municipality_file': next((p.name for p in files if '_Comuni.csv' in p.name), pd.NA),
        'province_file': next((p.name for p in files if '_Province.csv' in p.name), pd.NA),
        'region_file': next((p.name for p in files if '_Regioni.csv' in p.name), pd.NA),
    })

inventory = pd.DataFrame(inventory)
display(inventory)
assert inventory[['municipality_file','province_file','region_file']].notna().all().all()

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Municipality population panel

For the main market panel we use `Età = 999`, which represents the official municipality total. Reading only the required columns keeps the notebook substantially lighter than loading the full age-detail dataset into memory.


In [ ]:
def load_municipality_totals(year):
    path = RAW_DIR / str(year) / f'POSAS_{year}_it_Comuni.csv'
    df = pd.read_csv(
        path, sep=';', encoding='utf-8-sig',
        usecols=['Codice comune','Comune','Età','Totale maschi','Totale femmine','Totale'],
    )
    df.columns = ['municipality_code','municipality','age','male','female','population']
    df['municipality_code'] = df['municipality_code'].astype('string').str.strip()
    df['municipality'] = df['municipality'].astype('string').str.strip()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    totals = df.loc[df['age'].eq(999)].copy()
    if totals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate municipality total keys')
    totals['year'] = year
    return totals[['year','municipality_code','municipality','population']]

municipality_panel = pd.concat(
    [load_municipality_totals(year) for year in YEARS],
    ignore_index=True,
)
print(f'Municipality-year observations: {len(municipality_panel):,}')
display(municipality_panel.head())


## 3. Data-quality controls

The key analytical invariant is one observation per **municipality + year**. Missing and duplicate keys are treated as structural errors, not silently repaired.


In [ ]:
quality = pd.DataFrame({
    'metric': [
        'rows', 'unique municipality-year keys', 'missing municipality codes',
        'duplicate municipality-year keys', 'missing population',
        'negative population', 'years', 'municipalities in latest year'
    ],
    'value': [
        len(municipality_panel),
        municipality_panel[['year','municipality_code']].drop_duplicates().shape[0],
        municipality_panel['municipality_code'].isna().sum(),
        municipality_panel.duplicated(['year','municipality_code']).sum(),
        municipality_panel['population'].isna().sum(),
        (municipality_panel['population'] < 0).sum(),
        municipality_panel['year'].nunique(),
        municipality_panel.loc[municipality_panel['year'].eq(municipality_panel['year'].max()), 'municipality_code'].nunique(),
    ],
})
display(quality)
assert municipality_panel.duplicated(['year','municipality_code']).sum() == 0
assert municipality_panel['municipality_code'].notna().all()
assert municipality_panel['population'].notna().all()
assert (municipality_panel['population'] >= 0).all()


## 4. National population trend

National population is obtained by summing municipality totals. A mean across municipalities would not have the same economic interpretation.


In [ ]:
national_population = (
    municipality_panel.groupby('year', as_index=False)
    .agg(population=('population','sum'), municipalities=('municipality_code','nunique'))
)
national_population['yoy_pct'] = national_population['population'].pct_change() * 100
display(national_population)

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(national_population['year'], national_population['population'], marker='o')
ax.set_title('Italy — Population Trend')
ax.set_xlabel('Year')
ax.set_ylabel('Population')
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## 5. Municipality demographic dynamics

Year-over-year changes expose geographic heterogeneity hidden by the national series. Absolute and percentage changes are both retained because percentage growth can be extreme for very small municipalities.


In [ ]:
municipality_panel = municipality_panel.sort_values(['municipality_code','year']).reset_index(drop=True)
municipality_panel['population_change'] = municipality_panel.groupby('municipality_code')['population'].diff()
previous = municipality_panel.groupby('municipality_code')['population'].shift(1)
municipality_panel['population_change_pct'] = municipality_panel['population_change'].div(previous.where(previous.gt(0))).mul(100)

latest_year = municipality_panel['year'].max()
latest_dynamics = municipality_panel[municipality_panel['year'].eq(latest_year)].copy()

print('Largest absolute declines')
display(latest_dynamics.dropna(subset=['population_change']).sort_values('population_change').head(15))
print('Largest absolute increases')
display(latest_dynamics.dropna(subset=['population_change']).sort_values('population_change', ascending=False).head(15))


## 6. Population concentration

Population concentration provides a useful demographic context for subsequent real-estate analysis.


In [ ]:
national_latest = latest_dynamics['population'].sum()
top_municipalities = latest_dynamics.sort_values('population', ascending=False).head(20).copy()
top_municipalities['national_share_pct'] = top_municipalities['population'] / national_latest * 100
display(top_municipalities[['municipality','population','national_share_pct']])

fig, ax = plt.subplots(figsize=(10,7))
plot_data = top_municipalities.sort_values('population')
ax.barh(plot_data['municipality'], plot_data['population'])
ax.set_title(f'Top 20 Municipalities by Population — {latest_year}')
ax.set_xlabel('Population')
fig.tight_layout()
plt.show()


## 7. Regional and provincial context

Region and province files are already aggregated by geography. We use their `Età = 999` records for geographic rankings and avoid reconstructing regional mappings from municipality codes.


In [ ]:
def load_aggregate_totals(year, level):
    suffix = 'Province' if level == 'province' else 'Regioni'
    path = RAW_DIR / str(year) / f'POSAS_{year}_it_{suffix}.csv'
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    df.columns = [str(c).replace('\ufeff','').strip() for c in df.columns]
    df['Età'] = pd.to_numeric(df['Età'], errors='coerce')
    df['Totale'] = pd.to_numeric(df['Totale'], errors='coerce')
    return df.loc[df['Età'].eq(999)].copy()

latest_regions = load_aggregate_totals(latest_year, 'region')
latest_provinces = load_aggregate_totals(latest_year, 'province')

print('Largest regions')
display(latest_regions.sort_values('Totale', ascending=False).head(10))
print('Largest provinces')
display(latest_provinces.sort_values('Totale', ascending=False).head(10))


## 8. Regional age structure

Population size alone does not capture demographic composition. We classify ages into 0–14, 15–64 and 65+ to provide context for housing-demand analysis.


In [ ]:
path = RAW_DIR / str(latest_year) / f'POSAS_{latest_year}_it_Regioni.csv'
region_age = pd.read_csv(path, sep=';', encoding='utf-8-sig', usecols=['Regione','Età','Totale'])
region_age['Età'] = pd.to_numeric(region_age['Età'], errors='coerce')
region_age['Totale'] = pd.to_numeric(region_age['Totale'], errors='coerce')
region_age = region_age.loc[region_age['Età'].between(0,100)].copy()
region_age['age_group'] = pd.cut(region_age['Età'], bins=[-1,14,64,100], labels=['0-14','15-64','65+'])

region_age = (
    region_age.groupby(['Regione','age_group'], observed=True)['Totale']
    .sum().reset_index(name='population')
)
region_age['share_pct'] = region_age['population'] / region_age.groupby('Regione')['population'].transform('sum') * 100
age_structure = region_age.pivot(index='Regione', columns='age_group', values='share_pct').reset_index()
display(age_structure.sort_values('65+', ascending=False).head(15))


## 9. Analytical hand-off

### Reusable outputs
- `municipality_panel`: one row per municipality-year;
- `national_population`: national population trend and YoY change;
- `latest_dynamics`: latest population level and demographic change;
- `latest_regions` / `latest_provinces`: geographic demographic context;
- `age_structure`: regional demographic composition.

### Next integration

Join this population panel with the validated OMI quotation and transaction panels using a controlled **municipality + reference period/year** key. This enables questions such as whether population growth is associated with stronger price appreciation or transaction activity.

### Interpretation limits

Population change is a contextual indicator, not a causal explanation of real-estate prices or transaction volumes. Results must account for geography, market size, property mix and data coverage.
